In [3]:
import io
import json
import re
from typing import List, Dict, Any, Optional, Tuple

import gradio as gr
from PyPDF2 import PdfReader


# =========================
# 1) Checklist (PDF export aware)
# =========================

CHECKLIST = {
    "source": "WCAG 2.2–based Accessibility Checklist (PDF Export Aware)",
    "items": [
        # Auto-checkable from PDF
        {"id": "doc_title_present", "category": "Document",
         "description": "The PDF has a clear title in its file properties (WCAG 2.4.2 Page titled).", "weight": 1.0},
        {"id": "doc_language_present", "category": "Document",
         "description": "The PDF says what language it is written in (WCAG 3.1.1 Language of page).", "weight": 1.0},
        {"id": "doc_tagged_pdf", "category": "Document",
         "description": "The PDF includes accessibility ‘tags’ (structure info for screen readers).", "weight": 2.0},
        {"id": "doc_text_extractable", "category": "Document",
         "description": "Most pages contain real text (not just pictures of text).", "weight": 2.0},

        {"id": "text_plain_language", "category": "Text",
         "description": "The writing looks reasonably easy to read for the intended audience.", "weight": 1.0},
        {"id": "text_avoid_all_caps", "category": "Text",
         "description": "Avoids long ALL CAPS paragraphs (harder to read).", "weight": 1.0},
        {"id": "text_glossary_for_acronyms", "category": "Text",
         "description": "Acronyms/technical words are explained (or there is a glossary).", "weight": 0.8},
        {"id": "text_inclusive_language", "category": "Text",
         "description": "Instructions aren’t device-specific (e.g., not only “click”).", "weight": 0.8},

        {"id": "links_descriptive", "category": "Links",
         "description": "Links are clearly described (avoid “click here”).", "weight": 1.5},

        # Manual checks
        {"id": "text_minimum_font_size", "category": "Visual design",
         "description": "Text size is comfortable to read and works when zoomed in.", "weight": 1.5},
        {"id": "text_colour_contrast", "category": "Visual design",
         "description": "Text has good contrast against the background.", "weight": 2.0},
        {"id": "text_semantic_structure", "category": "Structure",
         "description": "Headings and lists follow a clear, logical structure.", "weight": 1.5},

        {"id": "images_alt_text", "category": "Images",
         "description": "Important images have a text description (alt text / caption / nearby explanation).", "weight": 2.0},
        {"id": "images_avoid_text_in_images", "category": "Images",
         "description": "Avoids using images that contain lots of important text.", "weight": 1.5},

        {"id": "media_captions", "category": "Audio / video",
         "description": "Videos have captions (subtitles).", "weight": 2.0},
        {"id": "media_transcripts", "category": "Audio / video",
         "description": "Audio/video has a transcript when needed.", "weight": 2.0},

        {"id": "keyboard_access", "category": "Interaction",
         "description": "Everything can be done using only a keyboard (no mouse required).", "weight": 2.0},
        {"id": "focus_order", "category": "Interaction",
         "description": "When you press Tab, focus moves in a sensible order.", "weight": 1.5},
        {"id": "consistent_navigation", "category": "Interaction",
         "description": "Navigation is consistent (menus/buttons stay in the same place).", "weight": 1.5},
    ],
}


# =========================
# 2) Supportive grading policy (more forgiving)
# =========================
# - partial earns most of the points
# - fail still earns a small amount (PDF exports can hide information)
# - unconfirmed manual items do NOT reduce the score (they’re excluded)
SCORE_MAP_SUPPORTIVE = {"pass": 1.0, "partial": 0.85, "fail": 0.20}

GRADE_BANDS = [
    (85, "A", "Excellent foundation"),
    (75, "B", "Good progress"),
    (65, "C", "On the right track"),
    (50, "D", "Getting started"),
    (0,  "E", "Needs more work"),
]


# =========================
# 3) Helpers
# =========================

def file_to_bytes(uploaded) -> Optional[bytes]:
    if uploaded is None:
        return None
    if isinstance(uploaded, (bytes, bytearray)):
        return bytes(uploaded)
    if isinstance(uploaded, str):  # filepath
        with open(uploaded, "rb") as f:
            return f.read()
    if isinstance(uploaded, dict) and "data" in uploaded:
        return uploaded["data"]
    if hasattr(uploaded, "read"):
        return uploaded.read()
    raise TypeError(f"Unsupported upload type: {type(uploaded)}")


def safe_get_pdf_root(reader: PdfReader):
    try:
        return reader.trailer.get("/Root")
    except Exception:
        return None


def extract_text_per_page(reader: PdfReader) -> List[str]:
    out: List[str] = []
    for p in reader.pages:
        try:
            out.append(p.extract_text() or "")
        except Exception:
            out.append("")
    return out


def estimate_image_count(reader: PdfReader) -> int:
    count = 0
    for page in reader.pages:
        try:
            res = page.get("/Resources") or {}
            xobj = res.get("/XObject") if isinstance(res, dict) else None
            if not xobj:
                continue
            xobj = xobj.get_object()
            for _, obj in xobj.items():
                try:
                    obj = obj.get_object()
                    if obj.get("/Subtype") == "/Image":
                        count += 1
                except Exception:
                    continue
        except Exception:
            continue
    return count


# =========================
# 4) Readability scoring (simple offline)
# =========================

def count_syllables(word: str) -> int:
    w = re.sub(r"[^a-zA-Z]", "", word)
    if not w:
        return 0
    w = w.lower()
    vowels = "aeiouy"
    syllables = 0
    prev_vowel = False
    for c in w:
        if c in vowels:
            if not prev_vowel:
                syllables += 1
            prev_vowel = True
        else:
            prev_vowel = False
    if w.endswith("e") and syllables > 1:
        syllables -= 1
    return max(syllables, 1)


def flesch_reading_ease(text: str) -> float:
    words = re.findall(r"\b\w+\b", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    if not words:
        return 0.0
    num_words = len(words)
    num_sentences = max(len(sentences), 1)
    syllables = sum(count_syllables(w) for w in words)
    asl = num_words / num_sentences
    asw = syllables / num_words
    return 206.835 - 1.015 * asl - 84.6 * asw


def flesch_kincaid_grade(text: str) -> float:
    words = re.findall(r"\b\w+\b", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    if not words:
        return 0.0
    num_words = len(words)
    num_sentences = max(len(sentences), 1)
    syllables = sum(count_syllables(w) for w in words)
    asl = num_words / num_sentences
    asw = syllables / num_words
    return 0.39 * asl + 11.8 * asw - 15.59


def readability_band(flesch: float) -> str:
    if flesch >= 90: return "Very easy"
    if flesch >= 80: return "Easy"
    if flesch >= 70: return "Fairly easy"
    if flesch >= 60: return "Plain English"
    if flesch >= 50: return "Fairly difficult"
    if flesch >= 30: return "Difficult"
    return "Very difficult"


# =========================
# 5) Feature extraction
# =========================

def extract_features(course_bytes: bytes, readability_char_cap: int = 50_000) -> Dict[str, Any]:
    reader = PdfReader(io.BytesIO(course_bytes))
    root = safe_get_pdf_root(reader)

    # Title (from PDF file properties / metadata)
    meta_title = ""
    try:
        meta = getattr(reader, "metadata", None)
        if meta and getattr(meta, "title", None):
            meta_title = str(meta.title or "").strip()
    except Exception:
        meta_title = ""

    # Language (PDF /Lang)
    doc_lang = ""
    try:
        if root and root.get("/Lang"):
            doc_lang = str(root.get("/Lang")).strip()
    except Exception:
        doc_lang = ""

    # Tagged PDF (very rough signal)
    is_tagged = False
    try:
        if root and root.get("/StructTreeRoot") is not None:
            is_tagged = True
    except Exception:
        is_tagged = False

    page_texts = extract_text_per_page(reader)
    num_pages = len(page_texts)
    pages_with_text = sum(1 for t in page_texts if len((t or "").strip()) >= 20)
    text_page_ratio = pages_with_text / max(num_pages, 1)

    full_text = "\n".join(page_texts)
    text_for_readability = full_text[:readability_char_cap]

    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", full_text) if p.strip()]

    def mostly_all_caps(par: str) -> bool:
        letters = [c for c in par if c.isalpha()]
        if not letters:
            return False
        upper = sum(1 for c in letters if c.isupper())
        return (upper / len(letters)) > 0.8

    caps_pars = [p for p in paragraphs if len(p) > 20 and mostly_all_caps(p)]
    all_caps_ratio = len(caps_pars) / max(len(paragraphs), 1)

    acronyms = sorted(set(re.findall(r"\b[A-Z]{3,}\b", full_text)))

    lower = full_text.lower()
    click_count = len(re.findall(r"\bclick\b", lower))
    agnostic_verbs = ["select", "tap", "choose", "press", "activate"]
    agnostic_count = sum(len(re.findall(rf"\b{re.escape(v)}\b", lower)) for v in agnostic_verbs)

    links_total = len(re.findall(r"https?://\S+|www\.\S+", full_text))
    click_here_count = len(re.findall(r"\bclick here\b", lower))

    flesch = flesch_reading_ease(text_for_readability)
    fk_grade = flesch_kincaid_grade(text_for_readability)

    image_count = estimate_image_count(reader)

    return {
        "meta_title": meta_title,
        "doc_lang": doc_lang,
        "is_tagged": is_tagged,
        "num_pages": num_pages,
        "pages_with_text": pages_with_text,
        "text_page_ratio": text_page_ratio,
        "all_caps_ratio": all_caps_ratio,
        "acronyms": acronyms,
        "click_count": click_count,
        "agnostic_count": agnostic_count,
        "agnostic_verbs": agnostic_verbs,
        "links_total": links_total,
        "click_here_count": click_here_count,
        "flesch": flesch,
        "fk_grade": fk_grade,
        "image_count": image_count,
        "readability_char_cap": readability_char_cap,
    }


# =========================
# 6) Evaluation + supportive scoring
# =========================

def pretty_label(item: Dict[str, Any]) -> str:
    return f"{item['category']}: {item['description']}"


def grade_from_score(score: float) -> Tuple[str, str]:
    for cutoff, letter, label in GRADE_BANDS:
        if score >= cutoff:
            return letter, label
    return "E", "Needs more work"


def compute_score_supportive(items: List[Dict[str, Any]], include_statuses: Optional[set] = None) -> Dict[str, Any]:
    if include_statuses is None:
        include_statuses = set(SCORE_MAP_SUPPORTIVE.keys())

    total_w = 0.0
    total_pts = 0.0

    for it in items:
        s = it["status"]
        if s in include_statuses:
            w = float(it["weight"])
            total_w += w
            total_pts += w * SCORE_MAP_SUPPORTIVE.get(s, 0.0)

    score = round((100.0 * total_pts / total_w), 1) if total_w > 0 else 0.0
    letter, label = grade_from_score(score)
    return {"score": score, "grade": letter, "grade_label": label, "denominator_weight": round(total_w, 2)}


def static_evaluate(checklist: Dict[str, Any], feat: Dict[str, Any]) -> Dict[str, Any]:
    out: List[Dict[str, Any]] = []
    for item in checklist["items"]:
        cid = item["id"]
        w = float(item["weight"])
        status = "manual"
        reason = ""

        if cid == "doc_title_present":
            t = (feat["meta_title"] or "").strip()
            if t and t.lower() not in {"untitled", "unknown"}:
                status = "pass"
                reason = f"Good: the PDF title is set to “{t}”."
            else:
                status = "fail"
                reason = "The PDF title is missing. This is usually an easy fix in the export settings."

        elif cid == "doc_language_present":
            lang = (feat["doc_lang"] or "").strip()
            if lang:
                status = "pass"
                reason = f"Good: the PDF language is set to “{lang}”."
            else:
                status = "fail"
                reason = "The PDF does not say what language it is. Screen readers may mispronounce words without this."

        elif cid == "doc_tagged_pdf":
            if feat["is_tagged"]:
                status = "pass"
                reason = "Good: the PDF looks like it includes accessibility tags (structure information)."
            else:
                status = "fail"
                reason = (
                    "No accessibility tags were detected. Tags help screen readers understand headings, reading order, and lists. "
                    "If possible, re-export as a ‘tagged PDF’ or ‘accessible PDF’."
                )

        elif cid == "doc_text_extractable":
            ratio = feat["text_page_ratio"]
            if ratio >= 0.8:
                status = "pass"
                reason = f"Good: most pages contain real text ({ratio*100:.1f}%)."
            elif ratio >= 0.5:
                status = "partial"
                reason = f"Mixed: only some pages contain real text ({ratio*100:.1f}%). Some pages may be images of text."
            else:
                status = "fail"
                reason = f"Most pages look like images of text ({ratio*100:.1f}% with real text). OCR or a better export may help."

        elif cid == "text_plain_language":
            f = feat["flesch"]
            g = feat["fk_grade"]
            band = readability_band(f)
            status = "pass" if f >= 55 else ("partial" if f >= 35 else "fail")
            reason = (
                f"Reading score: Ease ≈ {f:.1f} ({band}); Grade level ≈ {g:.1f}. "
                "This is only a helpful clue, not a strict rule."
            )

        elif cid == "text_avoid_all_caps":
            r = feat["all_caps_ratio"]
            if r == 0:
                status = "pass"
                reason = "Good: no long ALL CAPS paragraphs detected."
            elif r < 0.20:
                status = "partial"
                reason = f"Some ALL CAPS text detected (~{r*100:.1f}%). Consider switching long sections to normal sentence case."
            else:
                status = "fail"
                reason = f"A lot of ALL CAPS text detected (~{r*100:.1f}%). This can be harder to read."

        elif cid == "text_glossary_for_acronyms":
            acr = feat["acronyms"]
            if not acr:
                status = "not_applicable"
                reason = "No acronyms detected."
            else:
                status = "manual"
                reason = (
                    f"Acronyms detected (examples: {', '.join(acr[:8])}). "
                    "Please confirm they are explained the first time they appear, or listed in a glossary."
                )

        elif cid == "text_inclusive_language":
            click_ct = feat["click_count"]
            ag_ct = feat["agnostic_count"]
            if click_ct == 0 and ag_ct > 0:
                status = "pass"
                reason = "Good: instructions use device-neutral words (like select/choose) and do not rely on “click”."
            elif click_ct == 0 and ag_ct == 0:
                status = "manual"
                reason = "No clear instruction words detected. Quick manual check recommended."
            else:
                status = "partial"
                reason = (
                    f"Found “click” {click_ct} time(s). Device-neutral words were found {ag_ct} time(s). "
                    "Where possible, prefer select/choose/tap/press/activate so it works for mouse, touch, and keyboard users."
                )

        elif cid == "links_descriptive":
            total = feat["links_total"]
            ch = feat["click_here_count"]
            if total == 0 and ch == 0:
                status = "not_applicable"
                reason = "No links detected in the extracted text."
            elif ch == 0:
                status = "pass"
                reason = "Good: “click here” was not detected."
            else:
                status = "partial" if (total > 0 and ch < total) else "fail"
                reason = "“Click here” was detected. Links are clearer when they describe what they do (e.g., “Download the worksheet”)."

        else:
            status = "manual"
            reason = "This needs a human check (a PDF export cannot reliably measure this)."

        out.append({
            "id": cid,
            "category": item["category"],
            "description": item["description"],
            "status": status,
            "weight": w,
            "reason": reason
        })

    return {"items": out}


def top_next_steps(items: List[Dict[str, Any]], k: int = 3) -> List[Dict[str, Any]]:
    def key(it):
        status_rank = {"fail": 0, "partial": 1, "pass": 2, "manual": 3, "not_applicable": 4, "unconfirmed": 5}
        return (status_rank.get(it["status"], 9), -float(it["weight"]))
    actionable = [it for it in items if it["status"] in {"fail", "partial"}]
    actionable.sort(key=key)
    return actionable[:k]


def confidence_label(manual_done: int, manual_total: int) -> str:
    if manual_total <= 0:
        return "High (no manual checks needed)"
    ratio = manual_done / manual_total
    if ratio >= 0.67:
        return "High (you confirmed most items)"
    if ratio >= 0.34:
        return "Medium (some items confirmed)"
    return "Low (few items confirmed yet)"


def build_glossary_md() -> str:
    return (
        "### Plain-English glossary (quick explanations)\n"
        "- **PDF title / file properties / metadata:** Extra information saved inside the PDF (like the document’s name). "
        "Screen readers use it, and it helps users identify the document.\n"
        "- **PDF language:** A setting inside the PDF that tells assistive technology what language to use for pronunciation.\n"
        "- **Tagged PDF (accessibility tags):** Not ‘tags’ like social media. It means the PDF contains structure info "
        "(headings, reading order, lists). This helps screen readers read the PDF in the right order.\n"
        "- **Extractable text:** If you can highlight and copy text, it’s usually ‘real text’. If it’s a scanned image, "
        "a screen reader may struggle unless OCR was applied.\n"
        "- **Readability score:** A rough estimate of how complex the writing is. This is guidance only.\n"
        "- **Device-neutral instructions:** Instructions like “select/choose” work for mouse, keyboard, and touch. "
        "“Click” can be confusing for touch or keyboard users.\n"
    )


# =========================
# 7) Gradio callbacks
# =========================

def analyze_course(course_file):
    try:
        course_bytes = file_to_bytes(course_file)
    except Exception as e:
        return f"**Error reading PDF:** {e}", gr.update(choices=[], value=[]), "", ""

    if not course_bytes:
        return "**Please upload a PDF to begin.**", gr.update(choices=[], value=[]), "", ""

    try:
        feat = extract_features(course_bytes)
    except Exception as e:
        return f"**Could not analyse the PDF:** {e}", gr.update(choices=[], value=[]), "", ""

    evaluation = static_evaluate(CHECKLIST, feat)
    items = evaluation["items"]

    # Supportive automated score: only pass/partial/fail included
    auto = compute_score_supportive(items)

    passed = [it for it in items if it["status"] == "pass"]
    partial = [it for it in items if it["status"] == "partial"]
    failed = [it for it in items if it["status"] == "fail"]
    manual = [it for it in items if it["status"] == "manual"]
    na = [it for it in items if it["status"] == "not_applicable"]

    manual_labels = [pretty_label(it) for it in manual]
    label_to_id = {pretty_label(it): it["id"] for it in manual}

    next_steps = top_next_steps(items, k=3)

    lines: List[str] = []
    lines.append("## Automatic check results (from the PDF)")
    lines.append("")
    lines.append(f"- **Supportive score (automatic items only):** {auto['score']} / 100")
    lines.append(f"- **Grade:** {auto['grade']} — {auto['grade_label']}")
    lines.append(f"- **Checklist used:** {CHECKLIST['source']}")
    lines.append("")
    lines.append("### What we looked at in your PDF")
    lines.append(f"- **Number of pages:** {feat['num_pages']}")
    lines.append(f"- **Pages with real text:** {feat['pages_with_text']} ({feat['text_page_ratio']*100:.1f}%)")
    lines.append(f"- **Images detected (rough estimate):** {feat['image_count']}")
    lines.append(f"- **PDF title found:** {'Yes' if (feat['meta_title'] or '').strip() else 'No'}")
    lines.append(f"- **PDF language set:** {feat['doc_lang'] if (feat['doc_lang'] or '').strip() else 'No'}")
    lines.append(f"- **Tagged PDF detected:** {'Yes' if feat['is_tagged'] else 'No'}")
    lines.append("")
    lines.append(build_glossary_md())
    lines.append("")
    lines.append("### Reading difficulty (helpful clue)")
    lines.append(
        f"- **Ease score:** {feat['flesch']:.1f} ({readability_band(feat['flesch'])})\n"
        f"- **Estimated grade level:** {feat['fk_grade']:.1f}\n"
        "- Tip: this is not a strict accessibility pass/fail — it just helps you judge clarity."
    )
    lines.append("")
    lines.append("### Recommended next steps (highest impact first)")
    if next_steps:
        for it in next_steps:
            lines.append(f"- **{pretty_label(it)}**  \n  {it['reason']}")
    else:
        lines.append("- Nothing urgent detected from automatic checks.")
    lines.append("")
    lines.append("### ✅ Looks good (automatic checks)")
    lines.extend(["- None."] if not passed else [f"- **{pretty_label(it)}**  \n  {it['reason']}" for it in passed])
    lines.append("")
    lines.append("### ⚠️ Needs a small improvement (automatic checks)")
    lines.extend(["- None."] if not partial else [f"- **{pretty_label(it)}**  \n  {it['reason']}" for it in partial])
    lines.append("")
    lines.append("### ❌ Needs attention (automatic checks)")
    lines.extend(["- None."] if not failed else [f"- **{pretty_label(it)}**  \n  {it['reason']}" for it in failed])
    lines.append("")
    if na:
        lines.append("### ℹ️ Not found in this PDF (so we skipped it)")
        for it in na:
            lines.append(f"- **{pretty_label(it)}**  \n  {it['reason']}")
        lines.append("")
    lines.append("### Next: confirm the manual checklist")
    lines.append(
        "Some things can’t be measured reliably from a PDF export (for example colour contrast). "
        "Tick the items you have checked and confirmed. Unticked manual items are treated as **not confirmed yet** "
        "(they do not lower your score)."
    )

    summary_md = "\n".join(lines)

    state_obj = {
        "evaluation": evaluation,
        "label_to_id": label_to_id,
        "manual_total": len(manual_labels),
    }

    progress_md = f"**Manual checklist progress:** 0 / {len(manual_labels)} confirmed"
    return summary_md, gr.update(choices=manual_labels, value=[]), json.dumps(state_obj), progress_md


def update_manual_progress(manual_checked: List[str], evaluation_state: str):
    if not evaluation_state:
        return "**Manual checklist progress:** 0 / 0 confirmed"
    try:
        state = json.loads(evaluation_state)
        total = int(state.get("manual_total", 0))
    except Exception:
        total = 0
    selected = len(manual_checked or [])
    return f"**Manual checklist progress:** {selected} / {total} confirmed"


def finalize_grade(manual_checked: List[str], evaluation_state: str):
    if not evaluation_state:
        return "Please run **Analyze course** first."

    state = json.loads(evaluation_state)
    evaluation = state["evaluation"]
    label_to_id = state["label_to_id"]
    manual_total = int(state.get("manual_total", 0))

    selected_ids = {label_to_id[label] for label in (manual_checked or []) if label in label_to_id}
    manual_done = len(selected_ids)

    updated_items = []
    for it in evaluation["items"]:
        if it["status"] == "manual":
            updated = dict(it)
            if it["id"] in selected_ids:
                updated["status"] = "pass"
                updated["reason"] = "Confirmed in the manual checklist."
            else:
                updated["status"] = "unconfirmed"
                updated["reason"] = "Not confirmed yet (excluded from scoring)."
            updated_items.append(updated)
        else:
            updated_items.append(it)

    final = compute_score_supportive(updated_items)

    passed = [it for it in updated_items if it["status"] == "pass"]
    partial = [it for it in updated_items if it["status"] == "partial"]
    failed = [it for it in updated_items if it["status"] == "fail"]
    unconfirmed = [it for it in updated_items if it["status"] == "unconfirmed"]

    conf = confidence_label(manual_done, manual_total)
    next_steps = top_next_steps(updated_items, k=5)

    lines: List[str] = []
    lines.append("## Final report (after your manual confirmations)")
    lines.append("")
    lines.append(f"- **Supportive score (confirmed items only):** {final['score']} / 100")
    lines.append(f"- **Grade:** {final['grade']} — {final['grade_label']}")
    lines.append(f"- **Confidence level:** {conf}")
    lines.append("")
    lines.append("### What this means")
    lines.append(
        "- The score includes the automatic checks **plus** any manual items you confirmed.\n"
        "- Manual items you did not tick are treated as “not confirmed yet” (they do **not** reduce your score)."
    )
    lines.append("")
    lines.append("### Recommended next steps")
    if next_steps:
        for it in next_steps:
            lines.append(f"- **{pretty_label(it)}**  \n  {it['reason']}")
    else:
        lines.append("- No major issues detected in the scored items.")
    lines.append("")
    lines.append("### ✅ Met criteria")
    lines.extend(["- None."] if not passed else [f"- **{pretty_label(it)}**" for it in passed])
    lines.append("")
    lines.append("### ⚠️ Partially met criteria")
    lines.extend(["- None."] if not partial else [f"- **{pretty_label(it)}**" for it in partial])
    lines.append("")
    lines.append("### ❌ Needs attention")
    lines.extend(["- None."] if not failed else [f"- **{pretty_label(it)}**" for it in failed])
    lines.append("")
    lines.append("### 🕒 Not confirmed yet (manual items)")
    lines.append(f"- **{len(unconfirmed)}** item(s) not confirmed yet. Confirming more items increases confidence.")

    return "\n".join(lines)


# =========================
# 8) UI (same layout)
# =========================

custom_css = """
:root{
  --surface-bg: rgba(110, 86, 207, 0.30);
  --surface-border: rgba(110, 86, 207, 0.36);
  --surface-checked: rgba(110, 86, 207, 0.46);
  --surface-checked-border: rgba(110, 86, 207, 0.70);
  --ink: rgba(48, 33, 90, 0.92);
  --accent: rgb(110, 86, 207);
}

body, .gradio-container { background: #ffffff !important; }
.gradio-container, .gradio-container * { color: var(--ink) !important; }

.surface {
  background: var(--surface-bg) !important;
  border: 1px solid var(--surface-border) !important;
  border-radius: 18px !important;
  padding: 14px !important;
  box-shadow: none !important;
}

.surface .wrap,
.surface .gr-panel,
.surface .form,
.surface .input-container,
.surface .file-preview,
.surface .upload-box,
.surface .file-dropzone,
.surface [data-testid="file-upload"],
.surface [data-testid="dropzone"],
.surface .prose,
.surface .markdown,
.surface pre,
.surface code,
.surface * {
  background: transparent !important;
  box-shadow: none !important;
}

input[type="checkbox"], input[type="radio"] {
  accent-color: var(--accent) !important;
  transform: scale(1.25);
}

.surface-button button{
  width: 100% !important;
  border-radius: 18px !important;
  padding: 12px 16px !important;
  font-weight: 650 !important;
  border: 1px solid var(--surface-border) !important;
  background: var(--surface-bg) !important;
  color: var(--ink) !important;
  box-shadow: none !important;
}
.surface-button button:hover,
.surface-button button:active,
.surface-button button:focus{
  background: var(--surface-bg) !important;
  outline: none !important;
}

.checklist-surface label{
  display: flex !important;
  align-items: flex-start !important;
  gap: 12px !important;
  padding: 12px 12px !important;
  border-radius: 14px !important;
  border: 1px solid var(--surface-border) !important;
  margin: 10px 0 !important;
  background: transparent !important;
}

.checklist-surface label:has(input[type="checkbox"]:checked){
  background: var(--surface-checked) !important;
  border: 2px solid var(--surface-checked-border) !important;
}

.checklist-surface label:has(input[type="checkbox"]:checked) span{
  font-weight: 650 !important;
}

.gr-checkboxgroup, .gr-checkbox { font-size: 14px !important; line-height: 1.35 !important; }
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# Course Accessibility Checker (PDF)")
    gr.Markdown(
        "Upload a **PDF**. The tool will check what it can from the PDF itself, then you can confirm extra items manually."
    )

    with gr.Row():
        with gr.Column(scale=1, min_width=360):
            course_input = gr.File(
                label="Upload your PDF",
                file_types=[".pdf"],
                type="filepath",
                elem_classes=["surface"],
            )
            analyze_btn = gr.Button(
                "1) Run automatic checks",
                elem_classes=["surface-button"],
            )

        with gr.Column(scale=2):
            auto_summary = gr.Markdown(value="", elem_classes=["surface"])

    gr.Markdown("## Manual checklist (you confirm these)")
    gr.Markdown(
        "These items can’t be measured reliably from a PDF export. Tick the ones you have checked and confirmed. "
        "Unticked items are treated as **not confirmed yet** (they won’t reduce your score)."
    )

    manual_progress = gr.Markdown("**Manual checklist progress:** 0 / 0 confirmed", elem_classes=["surface"])

    with gr.Row():
        with gr.Column(scale=2):
            manual_checklist = gr.CheckboxGroup(
                label="Manual checklist",
                choices=[],
                value=[],
                interactive=True,
                elem_classes=["surface", "checklist-surface"],
            )
        with gr.Column(scale=1, min_width=300):
            final_btn = gr.Button(
                "2) Create final report",
                elem_classes=["surface-button"],
            )
            final_report = gr.Markdown(value="", elem_classes=["surface"])

    eval_state = gr.State("")

    analyze_btn.click(
        fn=analyze_course,
        inputs=[course_input],
        outputs=[auto_summary, manual_checklist, eval_state, manual_progress],
    )

    manual_checklist.change(
        fn=update_manual_progress,
        inputs=[manual_checklist, eval_state],
        outputs=[manual_progress],
    )

    final_btn.click(
        fn=finalize_grade,
        inputs=[manual_checklist, eval_state],
        outputs=[final_report],
    )

# Run:
demo.launch()


C:\Users\jithin.charls\AppData\Local\Temp\ipykernel_1488\2203210910.py:758: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css) as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
